# Module 6 — Probabilistic Learning with MLE and EM (Revised Version)

This revised version of Module 6 improves the probabilistic campus model by correcting an important semantic issue in the location design. In the earlier version, **VillagePark** had been used in a way that made it look like an athletic training area. In reality, VillagePark is better understood as a general social and transitional campus space, such as a central fountain or hangout area that can be used by many kinds of students. To fix this, the revised code adds a new main location called **Track**, which explicitly represents athletic training and practice behavior.

This correction matters because the probabilistic model should reflect the real meaning of campus spaces. If a location is given the wrong interpretation, then the learned probability distributions and latent-variable results can become misleading. By separating VillagePark from Track, the code creates a more realistic world model and gives the EM procedure a better chance of learning meaningful hidden patterns.

The module still focuses on the same overall goal: learning probabilistic models from observed campus data. It is organized into three parts:

1. **Statistical learning with MLE**
2. **Complete-data probabilistic learning**
3. **Hidden-variable learning with EM**

The focus subgroup for the hidden-variable section remains **Athlete at 16:00–17:30**, but this time the code models that period more carefully by allowing Track behavior to emerge explicitly.

---

## Reproducibility and Technical Setup

The code begins by importing the required packages and fixing a random seed for both Python’s `random` module and NumPy. This guarantees reproducibility across runs. Reproducibility is especially important here because the module depends on two major stochastic processes:
- random generation of the synthetic observed dataset,
- and random initialization of EM parameters.

The code also configures Matplotlib to use the `"Agg"` backend. This means plots are saved directly to files instead of being displayed interactively on screen. That is a useful implementation choice when running the script in environments where graphical display is unavailable, such as terminal-based execution or server environments.

---

## Canonical Project Schema

The first major section defines the project schema. This includes:
- the student types,
- the campus time slots,
- and the location hierarchy.

The schema is consistent with the rest of the campus project, but one important revision has been made: **Track** is now included as a separate main location and also as its own inner location.

This means the set of main locations now includes:
- LawrenceHall
- ThayerHall
- WestPenn
- StudentCenter
- Library
- VillagePark
- Track
- PlayHouse
- BoulevardStudios
- BoulevardAppartments
- OffCampus

The distinction between VillagePark and Track is central to the revised implementation. VillagePark is explicitly described as a general university gathering area, while Track represents training and practice behavior. This is a better conceptual design because it prevents the model from attributing all late-afternoon athletic activity to a socially shared location that should not function as a dedicated training site.

---

## Revised Synthetic Campus World

The next section defines the synthetic world used to generate observed data. As in previous modules, this world is built through weighted probability distributions. These weights serve as the hidden ground truth from which the observed dataset is sampled.

### Main-location distributions

The code defines `MAIN_LOCATION_WEIGHTS`, which specify:

\[
P(\text{main\_location} \mid \text{student\_type}, \text{time\_slot})
\]

for all student-type and time combinations.

The key revision appears in the athlete distributions:
- In earlier periods, athletes may still be OffCampus, in StudentCenter, or in academic buildings.
- Around **15:00–15:30**, Track begins to appear as a transition destination.
- At **16:00–17:30**, Track becomes the strongest athletic destination, replacing the earlier misuse of VillagePark.

This is a very important implementation change. It means that when the code later learns distributions or latent components for athletes in the late afternoon, it is no longer forced to interpret VillagePark as the athletic cluster. Instead, Track can emerge directly as the location most associated with training.

### Inner-location distributions

The code also defines `INNER_LOCATION_WEIGHTS`, which specify:

\[
P(\text{inner\_location} \mid \text{main\_location}, \text{student\_type})
\]

This section is also updated to include Track. Since Track is both a main location and its own single inner location, the corresponding conditional distribution is simply:

\[
P(\text{Track} \mid \text{Track}) = 1
\]

This makes the modeling cleaner, because athletic practice is no longer ambiguously represented through general public spaces.

---

## Sampling Functions and Data Generation

The implementation then defines helper functions to sample from the synthetic world.

### Weighted choice

The function `weighted_choice` takes a dictionary of categories and weights and performs a weighted random draw. This is the core mechanism that turns the probability design into actual observations.

### Main-location sampling

The function `sample_main_location` uses the main-location weights for the chosen student type and time slot to sample a building or campus area.

### Inner-location sampling

The function `sample_inner_location` uses the inner-location weights for the given student type and main location. If there is no exact weight specification, it falls back to a random valid room inside the selected main location.

### Dataset generation

The function `generate_dataset` creates the full observed dataset. For each sample, it randomly chooses:
- a student type,
- a time slot,
- a main location,
- and an inner location.

The resulting DataFrame contains four columns:
- `student_type`
- `time_slot`
- `main_location`
- `inner_location`

The code generates 30,000 observations. This gives enough data for stable probability estimation in Parts 1 and 2 and a sufficiently rich subgroup for EM in Part 3.

---

## Part 1 — Statistical Learning (20.1)

The first probabilistic learning task is simple maximum likelihood estimation for a categorical distribution.

### Chosen subgroup

The subgroup selected for Part 1 is:

- **Student type:** Athlete  
- **Time slot:** 07:00–08:00

This subgroup is isolated from the full dataset so that the model can focus on one well-defined probability distribution.

### Defining the model

For this subgroup, the random variable is the main location of an athlete in the early morning. If the possible observed locations are \(L_1, L_2, \dots, L_J\), then the model is a categorical distribution:

\[
\theta_j = P(L = L_j), \qquad \sum_j \theta_j = 1
\]

### Computing the MLE

Because the observations are fully observed, the maximum likelihood estimate is simply the empirical frequency of each location:

\[
\hat{\theta}_j = \frac{\text{count}(L_j)}{N}
\]

The code computes this by counting occurrences of each main location in the subgroup and dividing by the subgroup size.

### Interpreting the estimates

The code does something useful in this revised version: after computing the MLE probabilities, it prints an interpretation in plain language, for example:

- About \(p \times 100\%\) of athletes at 07:00–08:00 are in a given location.

This makes the output more readable and helps connect the formal probability distribution to a real-world interpretation of student behavior.

### Evaluating likelihood

The function `categorical_log_likelihood` computes the log-likelihood of the observed subgroup data under a given categorical distribution:

\[
\log P(L_1, \dots, L_N \mid \theta)
= \sum_{n=1}^N \log P(L_n \mid \theta)
\]

The code evaluates this log-likelihood under:
1. the MLE distribution,
2. and a uniform baseline over the observed support.

Since the MLE is the parameter setting that maximizes likelihood for the observed data, its log-likelihood should be higher than that of the uniform model. This provides a direct numerical demonstration of the value of maximum likelihood estimation.

### Why this part matters

This section establishes the fundamental probabilistic learning pattern:
- define a distribution,
- estimate it from observed data,
- and evaluate how well it explains the data.

It is the simplest form of probabilistic learning in the module, and it sets up the intuition for the more structured models that follow.

---

## Part 2 — Complete Data Learning (20.2)

The second section learns structured conditional probability distributions from the full observed dataset.

### Learning from complete observations

At this stage, all relevant variables are fully observed:
- student type,
- time slot,
- main location,
- inner location.

Because of that, the required probabilities can be estimated directly with normalized counts.

### Main-location conditional probabilities

The first learned table is:

\[
P(\text{main\_location} \mid \text{student\_type}, \text{time\_slot})
\]

The code groups the data by `student_type` and `time_slot`, and within each group computes normalized counts over `main_location`.

This produces a nested dictionary that allows queries such as:
- probability an athlete is in StudentCenter at 16:00–17:30,
- probability a Copa student is in PlayHouse at 08:00–09:30.

### Inner-location conditional probabilities

The second learned table is:

\[
P(\text{inner\_location} \mid \text{main\_location}, \text{student\_type}, \text{time\_slot})
\]

The code groups the data by:
- student type,
- time slot,
- and main location,

and then computes normalized counts of inner locations within each group.

This allows more detailed queries such as:
- probability an athlete is in Gym given that the athlete is in StudentCenter at 16:00–17:30,
- probability an athlete is in Track given that the athlete is in Track at 16:00–17:30.

### Example inference

The revised implementation includes two inference examples:
1. `StudentCenter > Gym`
2. `Track > Track`

for the subgroup Athlete at 16:00–17:30.

This is an excellent addition because it highlights the main semantic correction in the revised code. The model can now separately evaluate:
- support-oriented athletic behavior in StudentCenter,
- and explicit training behavior on Track.

The joint probability is computed as:

\[
P(\text{main}, \text{inner} \mid \text{type}, \text{time})
=
P(\text{main} \mid \text{type}, \text{time})
\cdot
P(\text{inner} \mid \text{main}, \text{type}, \text{time})
\]

This makes the inference procedure consistent with the structured probability factorization used in earlier modules.

### Why this part matters

This section shows how complete-data learning can be implemented entirely through grouped counts and normalization. It also demonstrates how the revised schema leads to more meaningful inference: Track is no longer hidden inside VillagePark behavior, so the system can explicitly assign probability mass to athletic practice.

---

## Part 3 — Hidden Variables and EM (20.3)

The third and most advanced section introduces hidden-variable learning.

### Revised observed variable

In the earlier version, the EM model used only the main location as the observed variable. In this revised implementation, the observed variable is made more detailed. The code constructs a **place token**:

\[
X = \text{main\_location} > \text{inner\_location}
\]

So instead of observing only broad locations such as StudentCenter or WestPenn, the model now observes tokens like:
- `StudentCenter > Gym`
- `Track > Track`
- `WestPenn > RegularClassroom`

This is an important improvement because it gives the latent-variable model more expressive detail. Different behaviors that would have been merged under the same main location can now be partially separated by their inner-location structure.

### Chosen EM subgroup

The subgroup used for EM is:

- **Student type:** Athlete  
- **Time slot:** 16:00–17:30

This is the same subgroup as before, but now the late-afternoon athletic behavior is modeled more realistically because Track exists as a separate location.

### Hidden variable

The hidden variable is still an unlabeled activity-mode variable with \(K=3\) latent components:

\[
Z \in \{0,1,2\}
\]

Although the code later interprets these components, EM itself does not assign semantic names. It only learns the mixture structure that best explains the observed place-token distribution.

### Mixture model

The model assumes that observed place tokens are generated from a categorical mixture:

\[
P(X) = \sum_z P(Z=z)\,P(X \mid Z=z)
\]

The parameters are:
- mixture weights \(\pi_z = P(Z=z)\),
- component distributions \(\phi_{z,m} = P(X=m \mid Z=z)\).

Because \(Z\) is hidden, these parameters cannot be estimated by direct counting, so EM is used.

---

## Anchor-Based Initialization

One of the strongest improvements in this revised implementation is the use of **anchor-based initialization**.

Instead of initializing all mixture components randomly without structure, the function `initialize_em_anchor_based` biases the initial components toward semantically meaningful anchors:
- Mode 0 is biased toward **Track**
- Mode 1 is biased toward **StudentCenter**
- Mode 2 is biased toward **WestPenn**

This is a smart implementation choice for two reasons.

First, it aligns the initialization with the corrected campus semantics. Since Track now explicitly represents athletic training, it makes sense to encourage one component to start near Track behavior.

Second, EM for mixture models can sometimes converge to poor local optima if initialization is not informative. By anchoring components near distinct behavior types, the algorithm is more likely to converge to interpretable solutions.

The code also adds small random noise to the initial component distributions to avoid exact symmetry and then normalizes the rows so that each component remains a valid categorical distribution.

---

## EM Implementation

The function `em_categorical_mixture` implements EM for a mixture of categorical distributions over place tokens.

### E-step

For each observation \(X_n\), the algorithm computes the posterior responsibility of each hidden component:

\[
\gamma_{nz} = P(Z=z \mid X_n)
= \frac{\pi_z \phi_{z,X_n}}{\sum_k \pi_k \phi_{k,X_n}}
\]

These responsibilities are stored in the matrix `gamma`.

### M-step

The mixture weights are updated using expected component counts:

\[
N_z = \sum_n \gamma_{nz}, \qquad
\pi_z = \frac{N_z}{N}
\]

The component distributions are updated using soft counts over observed tokens:

\[
\phi_{z,m} =
\frac{
\sum_{n: X_n = m} \gamma_{nz}
}{
\sum_n \gamma_{nz}
}
\]

This updates each component to reflect how strongly it is responsible for each token.

### Log-likelihood computation

After each iteration, the algorithm computes:

\[
\log P(X_1, \dots, X_N)
=
\sum_n \log\left(\sum_z \pi_z \phi_{z,X_n}\right)
\]

This is stored in `loglik_trace`.

The code requires at least a minimum number of iterations before allowing convergence, and then stops when the absolute improvement in log-likelihood falls below a very small tolerance. This helps prevent premature stopping.

### Why the implementation is strong

This EM implementation is more robust than a minimal textbook version because it includes:
- smarter initialization,
- a minimum iteration count,
- detailed convergence logging,
- and token-level observations rather than only building-level observations.

That makes the learned components more stable and more interpretable.

---

## Convergence Analysis and Saved Plots

The revised code saves two convergence plots instead of only displaying one:

1. **Raw log-likelihood**
2. **Log-likelihood improvement from iteration 1**

This is a very good analytical choice.

The raw log-likelihood plot shows whether EM is increasing the objective as expected. The improvement plot is even more informative because it highlights whether the algorithm is still making meaningful gains or has effectively stabilized.

Saving the plots as image files rather than displaying them directly is consistent with the earlier choice to use a non-interactive plotting backend. It also makes the results easier to include in reports or notebooks later.

---

## Post-hoc Interpretation of Hidden Modes

After the EM algorithm converges, the code prints:
- the final mixture weights,
- the final token distributions for each mode,
- and the most associated token for each mode.

Only after this does the code provide an interpretation of the modes.

A likely interpretation is:
- a mode concentrated on `Track > Track` corresponds to **Athletic Training / Practice**,
- a mode concentrated on `StudentCenter > Gym` or `StudentCenter > AthleteTrainers` corresponds to **Conditioning, Recovery, or Support**,
- a mode concentrated on `WestPenn > ...` corresponds to **Academic / Indoor activity**.

The code also makes a very important interpretive statement: VillagePark should be treated as a **social or transition area**, not as a dedicated training location.

This is exactly the conceptual correction that motivated the revised version. The hidden-mode interpretation is now much more faithful to campus meaning because the athletic-practice cluster can center on Track instead of being artificially forced into VillagePark.

---

## Why the Revision Improves the Model

This revised version improves the module in both conceptual and statistical terms.

### Conceptual improvement

The most important improvement is semantic correctness. Athletic training now has its own explicit location instead of being conflated with a general social area.

### Statistical improvement

Because the synthetic data is now better aligned with the intended meaning of locations, the learned distributions and latent components are also more meaningful. EM no longer has to explain late-afternoon athletic concentration using a misleading location. Instead, it can identify a component genuinely centered on Track behavior.

### Modeling improvement

The use of place tokens rather than only main locations gives the mixture model more resolution, and the anchor-based initialization makes convergence more stable and interpretable.

Together, these changes make the entire module stronger both as a probabilistic learning system and as a campus-behavior model.

---

## Reflective Questions

### How did hidden variables change learning?

Introducing hidden variables (the latent variable \( Z \), representing ActivityMode) fundamentally changes the learning process.

Before (Parts 1–2):
- All variables were **observed**
- Learning was straightforward:
  - Count frequencies
  - Normalize to get probabilities
- This is called **complete-data learning**

After introducing hidden variables (Part 3):
- We no longer observe the true cause (ActivityMode)
- We only observe outcomes (place tokens)
- This creates **uncertainty about which hidden state generated each observation**

As a result:
- Each data point is no longer assigned to a single category
- Instead, it is **softly assigned** to multiple hidden modes using probabilities (responsibilities \( \gamma \))
- Learning becomes:
  - **Inference + estimation combined**
  - We estimate both:
    - the hidden structure (ActivityMode)
    - the parameters of each mode

Hidden variables allow the model to discover underlying behavioral patterns that are not explicitly labeled in the data.

---

### Why can’t we directly maximize likelihood with missing data?

When hidden variables are present, the likelihood becomes:

\[
P(X) = \sum_z P(Z=z) P(X \mid Z=z)
\]

This creates a problem because the log-likelihood becomes:

\[
\log P(X) = \log \left( \sum_z \pi_z \phi_{z,X} \right)
\]

This expression is:
- Nonlinear and non-convex
- Difficult to optimize directly
- Lacks a closed-form solution

With complete data:
\[
\log P(X, Z) = \sum_n \log P(X_n \mid Z_n)
\]

But with missing \( Z \), we cannot assign observations to specific components, so simple counting (MLE) is not possible.

---

### How does EM guarantee improvement each iteration?

EM guarantees that the likelihood does not decrease at each iteration.

#### E-step (Expectation)
\[
\gamma_{nz} = P(Z=z \mid X_n)
\]

#### M-step (Maximization)
Update parameters using expected counts derived from \( \gamma \).

EM works by optimizing a lower bound on the log-likelihood. At each iteration:
- The E-step tightens the bound
- The M-step maximizes it

This ensures:
\[
\text{log-likelihood}_{t+1} \geq \text{log-likelihood}_t
\]

---

### When might EM converge to a local optimum?

EM may converge to a local optimum in several situations:

1. Poor initialization  
   - Bad initial parameters can lead to suboptimal convergence

2. Overlapping components  
   - If latent modes are not clearly separable, EM may mix them

3. Symmetry in the model  
   - Multiple equivalent solutions may exist

4. Limited or noisy data  
   - Insufficient signal to distinguish latent structure

In this project, overlap between locations like Track, StudentCenter, and WestPenn could lead to imperfect separation of modes.

Practical strategies include:
- Using informed initialization (as implemented)
- Running EM multiple times with different seeds
- Comparing final log-likelihood values

In [2]:
# ============================================================
# MODULE 6 — Probabilistic Learning with MLE and EM
# Fully revised version
#
# Key fix:
# - VillagePark is a general central hangout / fountain area
#   used by many students, NOT a dedicated athletic training space.
# - A new location "Track" is added so track-like athletic behavior
#   can be modeled explicitly instead of incorrectly using VillagePark.
#
# Focus subgroup for EM:
#   Athlete at 16:00-17:30
#
# Goal:
# Build and train probabilistic models using observed data.
#
# Covers:
# Part 1: Statistical Learning (20.1)
#   - define probability distribution
#   - compute MLE
#   - evaluate likelihood
#
# Part 2: Complete Data Learning (20.2)
#   - learn conditional probabilities from complete data
#   - perform inference
#
# Part 3: Hidden Variables and EM (20.3)
#   - introduce latent variable ActivityMode
#   - implement EM
#   - show convergence
# ============================================================

import math
import random
from collections import defaultdict

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

RANDOM_SEED = 7
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ============================================================
# 1. Canonical project schema
# ============================================================

STUDENT_TYPES = [
    "Regular Student",
    "Copa",
    "Athlete"
]

TIME_SLOTS = [
    "07:00-08:00",
    "08:00-09:30",
    "09:40-11:10",
    "11:20-12:50",
    "13:00-14:30",
    "15:00-15:30",
    "16:00-17:30",
    "18:00-21:00"
]

# Note:
# VillagePark is a central university gathering / fountain space used by many students.
# It is NOT a dedicated athletic training location.
# A separate "Track" location is included for athletic practice/training behavior.

LOCATION_HIERARCHY = {
    "LawrenceHall": [
        "DanceStudios",
        "DiningHall",
        "CopaTrainers",
        "HarryPotterRoom"
    ],
    "ThayerHall": [
        "RegularClassroom",
        "ComputerLab"
    ],
    "WestPenn": [
        "AnimationStudio",
        "ScreenWritingStudio",
        "RegularClassroom"
    ],
    "StudentCenter": [
        "Gym",
        "AthleteTrainers",
        "GameRoom",
        "CareerCenter"
    ],
    "Library": [
        "StudyRooms"
    ],
    "VillagePark": [
        "VillagePark"
    ],
    "Track": [
        "Track"
    ],
    "PlayHouse": [
        "ActingStudios",
        "FilmStudios"
    ],
    "BoulevardStudios": [
        "BoulevardStudios"
    ],
    "BoulevardAppartments": [
        "MusicalTheaterStudios"
    ],
    "OffCampus": [
        "OffCampus"
    ]
}

MAIN_LOCATIONS = list(LOCATION_HIERARCHY.keys())

# ============================================================
# 2. Synthetic campus world used to generate observed data
# ============================================================

MAIN_LOCATION_WEIGHTS = {
    "Athlete": {
        # Early morning athletes:
        # many are still off campus, some are in StudentCenter, a few indoors elsewhere
        "07:00-08:00": {
            "OffCampus": 5,
            "StudentCenter": 3,
            "LawrenceHall": 1,
            "ThayerHall": 1
        },

        "08:00-09:30": {
            "StudentCenter": 3,
            "LawrenceHall": 2,
            "Library": 2,
            "ThayerHall": 1,
            "VillagePark": 1,
            "OffCampus": 1
        },

        "09:40-11:10": {
            "StudentCenter": 4,
            "Library": 3,
            "ThayerHall": 2,
            "WestPenn": 1,
            "VillagePark": 1,
            "OffCampus": 1
        },

        "11:20-12:50": {
            "StudentCenter": 4,
            "LawrenceHall": 3,
            "BoulevardAppartments": 2,
            "Library": 2,
            "VillagePark": 1
        },

        "13:00-14:30": {
            "Library": 4,
            "ThayerHall": 3,
            "StudentCenter": 2,
            "WestPenn": 2,
            "VillagePark": 1
        },

        # Transition period before later afternoon activity
        "15:00-15:30": {
            "Track": 3,
            "StudentCenter": 3,
            "WestPenn": 2,
            "Library": 1,
            "VillagePark": 1
        },

        # Late afternoon athlete behavior:
        # now uses Track explicitly instead of incorrectly overloading VillagePark
        "16:00-17:30": {
            "Track": 5,
            "StudentCenter": 2,
            "WestPenn": 2,
            "OffCampus": 1,
            "VillagePark": 1
        },

        "18:00-21:00": {
            "StudentCenter": 4,
            "OffCampus": 3,
            "BoulevardAppartments": 2,
            "VillagePark": 1,
            "Track": 1
        }
    },

    "Copa": {
        "07:00-08:00": {
            "OffCampus": 10
        },
        "08:00-09:30": {
            "LawrenceHall": 3,
            "PlayHouse": 3,
            "BoulevardStudios": 3,
            "StudentCenter": 1,
            "OffCampus": 1
        },
        "09:40-11:10": {
            "BoulevardStudios": 4,
            "PlayHouse": 3,
            "LawrenceHall": 2,
            "StudentCenter": 2,
            "Library": 1
        },
        "11:20-12:50": {
            "LawrenceHall": 4,
            "StudentCenter": 3,
            "Library": 2,
            "PlayHouse": 2,
            "BoulevardStudios": 1
        },
        "13:00-14:30": {
            "BoulevardStudios": 4,
            "ThayerHall": 2,
            "PlayHouse": 3,
            "Library": 2,
            "StudentCenter": 1
        },
        "15:00-15:30": {
            "PlayHouse": 3,
            "StudentCenter": 2,
            "Library": 2,
            "BoulevardStudios": 3
        },
        "16:00-17:30": {
            "PlayHouse": 4,
            "StudentCenter": 3,
            "BoulevardStudios": 3,
            "Library": 1
        },
        "18:00-21:00": {
            "StudentCenter": 3,
            "OffCampus": 3,
            "BoulevardAppartments": 2,
            "PlayHouse": 2
        }
    },

    "Regular Student": {
        "07:00-08:00": {
            "OffCampus": 10
        },
        "08:00-09:30": {
            "ThayerHall": 4,
            "WestPenn": 3,
            "Library": 2,
            "StudentCenter": 1
        },
        "09:40-11:10": {
            "ThayerHall": 4,
            "WestPenn": 4,
            "Library": 2,
            "StudentCenter": 1
        },
        "11:20-12:50": {
            "LawrenceHall": 3,
            "StudentCenter": 3,
            "Library": 3,
            "ThayerHall": 1,
            "WestPenn": 1
        },
        "13:00-14:30": {
            "ThayerHall": 3,
            "WestPenn": 3,
            "Library": 3,
            "StudentCenter": 2
        },
        "15:00-15:30": {
            "StudentCenter": 3,
            "Library": 3,
            "ThayerHall": 2,
            "WestPenn": 2
        },
        "16:00-17:30": {
            "StudentCenter": 3,
            "Library": 3,
            "BoulevardAppartments": 2,
            "OffCampus": 2,
            "WestPenn": 1
        },
        "18:00-21:00": {
            "OffCampus": 4,
            "BoulevardAppartments": 3,
            "StudentCenter": 2,
            "Library": 1
        }
    }
}

INNER_LOCATION_WEIGHTS = {
    "Athlete": {
        "StudentCenter": {
            "Gym": 4,
            "AthleteTrainers": 3,
            "GameRoom": 1,
            "CareerCenter": 1
        },
        "LawrenceHall": {
            "DiningHall": 4,
            "HarryPotterRoom": 1,
            "DanceStudios": 1,
            "CopaTrainers": 1
        },
        "Library": {
            "StudyRooms": 1
        },
        "VillagePark": {
            "VillagePark": 1
        },
        "Track": {
            "Track": 1
        },
        "PlayHouse": {
            "ActingStudios": 1,
            "FilmStudios": 1
        },
        "BoulevardStudios": {
            "BoulevardStudios": 1
        },
        "BoulevardAppartments": {
            "MusicalTheaterStudios": 1
        },
        "ThayerHall": {
            "RegularClassroom": 2,
            "ComputerLab": 1
        },
        "WestPenn": {
            "RegularClassroom": 2,
            "AnimationStudio": 1,
            "ScreenWritingStudio": 1
        },
        "OffCampus": {
            "OffCampus": 1
        }
    },

    "Copa": {
        "LawrenceHall": {
            "DanceStudios": 4,
            "DiningHall": 2,
            "CopaTrainers": 3,
            "HarryPotterRoom": 1
        },
        "PlayHouse": {
            "ActingStudios": 4,
            "FilmStudios": 3
        },
        "BoulevardStudios": {
            "BoulevardStudios": 1
        },
        "StudentCenter": {
            "CareerCenter": 2,
            "GameRoom": 2,
            "Gym": 1,
            "AthleteTrainers": 1
        },
        "BoulevardAppartments": {
            "MusicalTheaterStudios": 1
        },
        "Library": {
            "StudyRooms": 1
        },
        "ThayerHall": {
            "RegularClassroom": 1,
            "ComputerLab": 1
        },
        "WestPenn": {
            "AnimationStudio": 2,
            "ScreenWritingStudio": 3,
            "RegularClassroom": 1
        },
        "VillagePark": {
            "VillagePark": 1
        },
        "Track": {
            "Track": 1
        },
        "OffCampus": {
            "OffCampus": 1
        }
    },

    "Regular Student": {
        "ThayerHall": {
            "RegularClassroom": 4,
            "ComputerLab": 3
        },
        "WestPenn": {
            "RegularClassroom": 4,
            "AnimationStudio": 1,
            "ScreenWritingStudio": 1
        },
        "StudentCenter": {
            "GameRoom": 3,
            "CareerCenter": 2,
            "Gym": 1,
            "AthleteTrainers": 1
        },
        "Library": {
            "StudyRooms": 1
        },
        "LawrenceHall": {
            "DiningHall": 3,
            "HarryPotterRoom": 1,
            "DanceStudios": 1,
            "CopaTrainers": 1
        },
        "BoulevardAppartments": {
            "MusicalTheaterStudios": 1
        },
        "VillagePark": {
            "VillagePark": 1
        },
        "Track": {
            "Track": 1
        },
        "PlayHouse": {
            "ActingStudios": 1,
            "FilmStudios": 1
        },
        "BoulevardStudios": {
            "BoulevardStudios": 1
        },
        "OffCampus": {
            "OffCampus": 1
        }
    }
}


def weighted_choice(weight_dict: dict[str, float], rng: random.Random) -> str:
    items = [(k, v) for k, v in weight_dict.items() if v > 0]
    total = sum(v for _, v in items)
    r = rng.uniform(0, total)
    upto = 0.0
    for key, w in items:
        upto += w
        if upto >= r:
            return key
    return items[-1][0]


def sample_main_location(student_type: str, time_slot: str, rng: random.Random) -> str:
    weights = MAIN_LOCATION_WEIGHTS[student_type][time_slot]
    return weighted_choice(weights, rng)


def sample_inner_location(student_type: str, time_slot: str, main_location: str, rng: random.Random) -> str:
    if student_type in INNER_LOCATION_WEIGHTS and main_location in INNER_LOCATION_WEIGHTS[student_type]:
        weights = INNER_LOCATION_WEIGHTS[student_type][main_location]
        return weighted_choice(weights, rng)
    return rng.choice(LOCATION_HIERARCHY[main_location])


def generate_dataset(n_samples: int = 30000, seed: int = 7) -> pd.DataFrame:
    rng = random.Random(seed)
    rows = []

    for _ in range(n_samples):
        student_type = rng.choice(STUDENT_TYPES)
        time_slot = rng.choice(TIME_SLOTS)

        main_location = sample_main_location(student_type, time_slot, rng)
        inner_location = sample_inner_location(student_type, time_slot, main_location, rng)

        rows.append({
            "student_type": student_type,
            "time_slot": time_slot,
            "main_location": main_location,
            "inner_location": inner_location
        })

    return pd.DataFrame(rows)


df = generate_dataset(n_samples=30000, seed=RANDOM_SEED)

print("Dataset shape:", df.shape)
print(df.head())

# ============================================================
# PART 1 — Statistical Learning (20.1)
# ============================================================

print("\n" + "=" * 70)
print("PART 1 — Statistical Learning (20.1)")
print("=" * 70)

subgroup_type = "Athlete"
subgroup_time = "07:00-08:00"

df_sub = df[
    (df["student_type"] == subgroup_type) &
    (df["time_slot"] == subgroup_time)
].copy()

print(f"\nSubgroup used for Part 1: {subgroup_type} at {subgroup_time}")
print("Subgroup size:", len(df_sub))

counts_sub = df_sub["main_location"].value_counts().sort_index()
print("\nObserved location counts:")
print(counts_sub)

N_sub = len(df_sub)
theta_hat = (counts_sub / N_sub).to_dict()

print("\nMLE estimates P(Location | Athlete, 07:00-08:00):")
for loc, p in theta_hat.items():
    print(f"  {loc:15s}: {p:.4f}")

print("\nInterpretation of Part 1 results:")
for loc, p in theta_hat.items():
    print(f"  About {p * 100:.1f}% of athletes at 07:00-08:00 are in {loc}.")


def categorical_log_likelihood(
    observations: list[str],
    probs: dict[str, float],
    eps: float = 1e-12
) -> float:
    ll = 0.0
    for x_i in observations:
        ll += math.log(probs.get(x_i, eps) + eps)
    return ll


obs_list = df_sub["main_location"].tolist()
ll_mle = categorical_log_likelihood(obs_list, theta_hat)

print(f"\nLog-likelihood of observed subgroup data under MLE parameters: {ll_mle:.3f}")

uniform_probs = {loc: 1 / len(theta_hat) for loc in theta_hat}
ll_uniform = categorical_log_likelihood(obs_list, uniform_probs)

print(f"Log-likelihood under uniform distribution: {ll_uniform:.3f}")
print("\nMLE should have higher likelihood than the uniform baseline.")

# ============================================================
# PART 2 — Complete Data Learning (20.2)
# ============================================================

print("\n" + "=" * 70)
print("PART 2 — Complete Data Learning (20.2)")
print("=" * 70)

main_conditional: dict[str, dict[str, dict[str, float]]] = defaultdict(lambda: defaultdict(dict))
grouped_main = df.groupby(["student_type", "time_slot"])

for (stype, time_slot), group in grouped_main:
    probs = group["main_location"].value_counts(normalize=True).to_dict()
    main_conditional[stype][time_slot] = probs

inner_conditional: dict[str, dict[str, dict[str, dict[str, float]]]] = defaultdict(
    lambda: defaultdict(lambda: defaultdict(dict))
)
grouped_inner = df.groupby(["student_type", "time_slot", "main_location"])

for (stype, time_slot, main_loc), group in grouped_inner:
    probs = group["inner_location"].value_counts(normalize=True).to_dict()
    inner_conditional[stype][time_slot][main_loc] = probs

example_type = "Athlete"
example_time = "16:00-17:30"
example_main = "StudentCenter"
example_inner = "Gym"

p_main = main_conditional[example_type][example_time].get(example_main, 0.0)
p_inner_given_main = inner_conditional[example_type][example_time][example_main].get(example_inner, 0.0)
p_joint = p_main * p_inner_given_main

print("\nInference example:")
print(f"P(main={example_main} | {example_type}, {example_time}) = {p_main:.4f}")
print(f"P(inner={example_inner} | main={example_main}, {example_type}, {example_time}) = {p_inner_given_main:.4f}")
print(f"P(main={example_main}, inner={example_inner} | {example_type}, {example_time}) = {p_joint:.4f}")

example_main2 = "Track"
example_inner2 = "Track"

p_main2 = main_conditional[example_type][example_time].get(example_main2, 0.0)
p_inner_given_main2 = inner_conditional[example_type][example_time][example_main2].get(example_inner2, 0.0)
p_joint2 = p_main2 * p_inner_given_main2

print("\nSecond inference example:")
print(f"P(main={example_main2} | {example_type}, {example_time}) = {p_main2:.4f}")
print(f"P(inner={example_inner2} | main={example_main2}, {example_type}, {example_time}) = {p_inner_given_main2:.4f}")
print(f"P(main={example_main2}, inner={example_inner2} | {example_type}, {example_time}) = {p_joint2:.4f}")

# ============================================================
# PART 3 — Hidden Variables and EM (20.3)
# Observed variable = place token = main_location > inner_location
# ============================================================

print("\n" + "=" * 70)
print("PART 3 — Hidden Variables and EM (20.3)")
print("=" * 70)

em_type = "Athlete"
em_time = "16:00-17:30"

df_em = df[
    (df["student_type"] == em_type) &
    (df["time_slot"] == em_time)
].copy()

df_em["place_token"] = df_em["main_location"] + " > " + df_em["inner_location"]

tokens = sorted(df_em["place_token"].unique())
obs_tokens = df_em["place_token"].tolist()
N_em = len(obs_tokens)

print(f"\nEM subgroup: {em_type} at {em_time}")
print("Subgroup size:", N_em)
print("Observed support:")
for tok in tokens:
    print(" ", tok)

print("\nObserved counts:")
print(df_em["place_token"].value_counts())

token_to_idx = {tok: i for i, tok in enumerate(tokens)}
x = np.array([token_to_idx[tok] for tok in obs_tokens])

K = 3
M = len(tokens)


def normalize_rows(mat: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    row_sums = mat.sum(axis=1, keepdims=True) + eps
    return mat / row_sums


def initialize_em_anchor_based(tokens: list[str], seed: int = 7) -> tuple[np.ndarray, np.ndarray]:
    """
    Smarter initialization:
    - Mode 0 biased toward Track
    - Mode 1 biased toward StudentCenter
    - Mode 2 biased toward WestPenn
    This aligns better with the corrected campus semantics.
    """
    rng = np.random.default_rng(seed)

    pi = np.array([1 / 3, 1 / 3, 1 / 3], dtype=float)
    phi = np.full((3, len(tokens)), 0.05, dtype=float)

    for j, tok in enumerate(tokens):
        if tok.startswith("Track"):
            phi[0, j] += 2.0
        if tok.startswith("StudentCenter"):
            phi[1, j] += 2.0
        if tok.startswith("WestPenn"):
            phi[2, j] += 2.0
        if tok.startswith("OffCampus"):
            phi[1, j] += 0.5
        if tok.startswith("VillagePark"):
            phi[1, j] += 0.25  # allow social/transition behavior to sit closer to support/transition mode

    phi += rng.random(phi.shape) * 0.01
    phi = normalize_rows(phi)
    return pi, phi


def compute_log_likelihood(x: np.ndarray, pi: np.ndarray, phi: np.ndarray, eps: float = 1e-12) -> float:
    ll = 0.0
    for xi in x:
        p = 0.0
        for z in range(len(pi)):
            p += pi[z] * phi[z, xi]
        ll += math.log(p + eps)
    return ll


def em_categorical_mixture(
    x: np.ndarray,
    K: int,
    M: int,
    tokens: list[str],
    max_iter: int = 200,
    min_iter: int = 10,
    tol: float = 1e-10,
    seed: int = 7
) -> tuple[np.ndarray, np.ndarray, list[float], np.ndarray]:
    """
    EM for a mixture of categorical distributions.
    Observed variable = token index.
    """
    pi, phi = initialize_em_anchor_based(tokens, seed=seed)
    loglik_trace: list[float] = []

    for it in range(1, max_iter + 1):
        gamma = np.zeros((len(x), K), dtype=float)

        # E-step
        for n, xi in enumerate(x):
            numerators = pi * phi[:, xi]
            denom = numerators.sum() + 1e-12
            gamma[n, :] = numerators / denom

        # M-step
        Nk = gamma.sum(axis=0)
        pi = Nk / len(x)

        phi = np.zeros((K, M), dtype=float)
        for z in range(K):
            for m in range(M):
                phi[z, m] = gamma[x == m, z].sum()
            phi[z, :] /= (phi[z, :].sum() + 1e-12)

        ll = compute_log_likelihood(x, pi, phi)
        loglik_trace.append(ll)

        if it == 1 or it % 5 == 0:
            print(f"iter={it:>3}  loglik={ll:.6f}")

        if it >= min_iter and len(loglik_trace) > 1:
            improvement = abs(loglik_trace[-1] - loglik_trace[-2])
            if improvement < tol:
                break

    return pi, phi, loglik_trace, gamma


pi_hat, phi_hat, ll_trace, gamma = em_categorical_mixture(
    x=x,
    K=K,
    M=M,
    tokens=tokens,
    max_iter=200,
    min_iter=10,
    tol=1e-10,
    seed=RANDOM_SEED
)

print("\nFinal mixture weights P(ActivityMode):")
for z in range(K):
    print(f"  Mode {z}: {pi_hat[z]:.4f}")

print("\nFinal component distributions P(PlaceToken | ActivityMode):")
for z in range(K):
    print(f"\nMode {z}:")
    pairs = [(tokens[m], phi_hat[z, m]) for m in range(M)]
    pairs.sort(key=lambda t: t[1], reverse=True)
    for tok, p in pairs:
        print(f"  {tok:35s}: {p:.4f}")

print(f"\nFinal log-likelihood: {ll_trace[-1]:.6f}")

# ============================================================
# Convergence plots
# ============================================================

ll0 = ll_trace[0]
ll_improvement = [ll - ll0 for ll in ll_trace]

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(ll_trace) + 1), ll_trace, marker="o")
plt.title("EM Convergence — Raw Log-Likelihood")
plt.xlabel("Iteration")
plt.ylabel("Log-Likelihood")
plt.grid(True)
plt.tight_layout()
plt.savefig("module6_em_raw_loglikelihood.png", dpi=300, bbox_inches="tight")
plt.close()

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(ll_improvement) + 1), ll_improvement, marker="o")
plt.title("EM Convergence — Improvement from Iteration 1")
plt.xlabel("Iteration")
plt.ylabel("Log-Likelihood Improvement")
plt.grid(True)
plt.tight_layout()
plt.savefig("module6_em_loglikelihood_improvement.png", dpi=300, bbox_inches="tight")
plt.close()

print("\nSaved convergence plots:")
print("  module6_em_raw_loglikelihood.png")
print("  module6_em_loglikelihood_improvement.png")

# ============================================================
# Post-hoc interpretation of hidden modes
# ============================================================

print("\nPost-hoc interpretation of learned modes:")
for z in range(K):
    top_token = tokens[np.argmax(phi_hat[z])]
    print(f"  Mode {z} is most associated with: {top_token}")

print("""
Possible interpretation for Athlete at 16:00-17:30:
- A mode concentrated on Track tokens likely corresponds to Athletic Training / Practice.
- A mode concentrated on StudentCenter > Gym / AthleteTrainers may correspond to Conditioning, Recovery, or Support activity.
- A mode concentrated on WestPenn tokens may correspond to Academic / Indoor activity.
- VillagePark should be interpreted as social / transition campus presence rather than dedicated training.
These labels are interpretive; EM itself learns unlabeled latent components.
""".strip())

# ============================================================
# Markdown-ready explanations
# ============================================================

print("\n\n--- Markdown-ready explanation (paste into notebook) ---\n")
print(r"""
### Part 1 (Statistical Learning)
We define a categorical probability distribution over main campus locations for a fixed subgroup:
- Student type: **Athlete**
- Time slot: **07:00-08:00**

If the observed locations are \(L_1, \dots, L_N\), the model parameters are:

\[
\theta_j = P(L = \text{location}_j), \quad \sum_j \theta_j = 1
\]

The maximum likelihood estimate is the empirical frequency:

\[
\hat{\theta}_j = \frac{\text{count of location } j}{N}
\]

For this revised campus world, the early-morning athlete subgroup is designed so that many athletes are still OffCampus, some are already in StudentCenter, and a smaller fraction are indoors elsewhere. This better reflects the corrected interpretation that VillagePark is a general student gathering area rather than an athletic training location.

We also compute the log-likelihood of the observed data under the estimated parameters and compare it to a uniform baseline.

### Part 2 (Complete Data Learning)
We learn the following conditional probabilities from complete observations:
- \(P(\text{main_location} \mid \text{student_type}, \text{time_slot})\)
- \(P(\text{inner_location} \mid \text{main_location}, \text{student_type}, \text{time_slot})\)

Because all variables are observed in the synthetic dataset, parameter learning is done by normalized counts.

Inference for frontend queries uses:

\[
P(\text{main}, \text{inner} \mid \text{type}, \text{time})
=
P(\text{main} \mid \text{type}, \text{time})
\cdot
P(\text{inner} \mid \text{main}, \text{type}, \text{time})
\]

This supports queries such as:
- probability an Athlete is in StudentCenter > Gym at 16:00-17:30
- probability an Athlete is in Track > Track at 16:00-17:30

### Part 3 (Hidden Variables and EM)
For the subgroup **Athlete at 16:00-17:30**, we introduce a latent variable:

\[
Z \in \{0,1,2\}
\]

The observed variable is a more detailed place token:

\[
X = \text{main_location} > \text{inner_location}
\]

We model the observations as a mixture:

\[
P(X) = \sum_z P(Z=z) P(X \mid Z=z)
\]

Since \(Z\) is hidden, we use the EM algorithm.

#### E-step
We compute the posterior responsibility of each hidden mode for each observation:

\[
\gamma_{nz} = P(Z=z \mid X_n)
= \frac{\pi_z \phi_{z,X_n}}{\sum_k \pi_k \phi_{k,X_n}}
\]

where:
- \(\pi_z = P(Z=z)\)
- \(\phi_{z,m} = P(X=m \mid Z=z)\)

#### M-step
We update parameters using expected counts:

\[
\pi_z = \frac{N_z}{N}, \quad N_z = \sum_n \gamma_{nz}
\]

\[
\phi_{z,m} = \frac{\sum_{n: X_n = m} \gamma_{nz}}{\sum_n \gamma_{nz}}
\]

#### Convergence analysis
We track:
- raw log-likelihood
- log-likelihood improvement from iteration 1

A nondecreasing log-likelihood indicates that EM is converging properly.

### Interpretation of hidden modes
The hidden modes are unlabeled components learned from data. After training, they can be interpreted post hoc as:
- athletic training / practice
- conditioning / recovery / support
- academic / indoor activity

VillagePark should be interpreted as a social / transition area, not as a dedicated training location.

### Model design summary
- Part 1: categorical distribution + MLE
- Part 2: complete-data conditional probability learning
- Part 3: latent-variable mixture model over detailed place tokens, fit with EM

This creates a coherent probabilistic learning system for the campus project.
""".strip())

Dataset shape: (30000, 4)
      student_type    time_slot  main_location inner_location
0             Copa  09:40-11:10      PlayHouse  ActingStudios
1          Athlete  08:00-09:30   LawrenceHall     DiningHall
2          Athlete  11:20-12:50  StudentCenter            Gym
3  Regular Student  11:20-12:50   LawrenceHall     DiningHall
4          Athlete  08:00-09:30      OffCampus      OffCampus

PART 1 — Statistical Learning (20.1)

Subgroup used for Part 1: Athlete at 07:00-08:00
Subgroup size: 1246

Observed location counts:
main_location
LawrenceHall     116
OffCampus        648
StudentCenter    369
ThayerHall       113
Name: count, dtype: int64

MLE estimates P(Location | Athlete, 07:00-08:00):
  LawrenceHall   : 0.0931
  OffCampus      : 0.5201
  StudentCenter  : 0.2961
  ThayerHall     : 0.0907

Interpretation of Part 1 results:
  About 9.3% of athletes at 07:00-08:00 are in LawrenceHall.
  About 52.0% of athletes at 07:00-08:00 are in OffCampus.
  About 29.6% of athletes at 07:0